Gerne\! Hier ist der Code im Google Colab Format mit erklärenden Textzellen. Ich habe angenommen, dass die Datei `ehrapy_input_index_ops.csv` im Colab-Umfeld verfügbar ist (z. B. durch Hochladen oder Mounten von Google Drive).

## 🩺 Google Colab: Ehrapy-Analyse und ML-Pipeline (Operationsebene)

Dieses Notebook demonstriert den Einsatz der `ehrapy`-Bibliothek zur Datenvorbereitung und Analyse medizinischer Daten, gefolgt von der Erstellung und Bewertung einer Scikit-Learn Machine-Learning-Pipeline (Random Forest) zur Vorhersage von Akutem Nierenversagen (AKI).

### Setup und Importe

Importiert werden die notwendigen Bibliotheken wie `os`, `numpy`, `pandas`, `matplotlib`, `seaborn` und `ehrapy` für die Datenverarbeitung. Zusätzlich werden die erforderlichen Module aus `sklearn` für das Machine Learning (Splitting, Preprocessing, Modell, Metriken) importiert.

Dieses Notebook ist fuer Kliniker, um Ihnen zu zeigen, wie Sie ohne Programmierkenntnisse mit ehrapy Daten analysieren koennen. Wir verwenden Google Colab, weil der Code hier ohne Installation von zusaetzlicher Software auf dem Computer mit Interzugang einfach zu verwenden ist.

In [1]:
# @title Code hier uberspringen und nicht ändern. Hier laden wir Module runter
! pip install ehrapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 6.5 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2


In [3]:
# Setup und Importe
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ehrapy as ep

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

### 1\) Daten in ehrapy laden und Feature-Typen korrigieren

Die Daten werden mithilfe von `ep.io.read_csv` in ein `anndata`-Objekt (`adata`) geladen, welches das primäre Datenformat in `ehrapy` ist.

Anschließend wird `ep.ad.infer_feature_types(adata)` verwendet, um die Datentypen der Spalten automatisch zu erkennen. Einige Spalten werden manuell korrigiert, um sicherzustellen, dass sie als `numeric` oder `categorical` (kategorial) für die nachfolgende Verarbeitung korrekt interpretiert werden.

In [5]:
# =========================
# 1) Daten in ehrapy laden
# =========================
try:
    adata = ep.io.read_csv(EH_CSV)
except FileNotFoundError:
    print(f"❌ Fehler: Datei '{EH_CSV}' nicht gefunden. Bitte laden Sie die Datei hoch.")
    # Platzhalter-adata für den Code-Fluss, falls das Laden fehlschlägt
    adata = None

if adata is not None:
    # Feature-Typen bestimmen (Info) und ggf. korrigieren
    ep.ad.infer_feature_types(adata)
    to_fix = {
        "n_ops": "numeric",           # deine ehrapy-Version nutzt 'numeric'/'categorical'/'date'
        "AKI_any_0_7": "categorical",
        "highest_AKI_stage_0_7": "categorical",
    }
    for feat, ftype in to_fix.items():
        if feat in adata.to_df().columns:
            try:
                ep.ad.replace_feature_types(adata, feat, ftype)
            except TypeError:
                ep.ad.replace_feature_types(adata=adata, feature=feat, corrected_type=ftype)

    print("Featuretypen (nach Korrektur):", adata.uns.get("feature_types", {}))

NameError: name 'EH_CSV' is not defined

In [ ]:
# =========================
# Einstellungen / Pfade
# =========================
# **WICHTIG:** Geben Sie hier den Pfad zu Ihrer CSV-Datei an.
# Beispiel für direkten Upload: "ehrapy_input_index_ops.csv"
# Beispiel für Google Drive: "/content/drive/MyDrive/IhrOrdner/ehrapy_input_index_ops.csv"
EH_CSV = input("Bitte geben Sie den Pfad zu Ihrer ehrapy_input_index_ops.csv-Datei ein: ")
os.makedirs("Diagramme", exist_ok=True)

### 2\) ML-Setup: Prädiktoren und Label definieren

In diesem Abschnitt werden die Daten für das Machine Learning vorbereitet:

1.  Das Label (`y`, die Zielvariable **AKI\_any\_0\_7**) wird extrahiert.
2.  Spalten, die zu einem Daten-Leak führen könnten (wie das Label selbst oder dessen Stufe), werden aus den Features (`X`) entfernt.
3.  Die gewünschten numerischen (`num_cols`) und kategorialen (`cat_cols`) Prädiktoren werden definiert und ausgewählt.
4.  Numerische Spalten werden explizit in einen numerischen Typ konvertiert (Fehler werden als `NaN` behandelt).
5.  Unerwünschte Spalten vom Typ `object` (die nicht zu den kategorialen Features gehören) werden entfernt.

<!-- end list -->

In [ ]:
# =========================================
# 2) ML-Setup: Leak vermeiden, Datentypen
# =========================================
if adata is not None:
    df_ml = adata.to_df()
    # Label früh puffern (um es später in obs zu speichern)
    label_series = df_ml["AKI_any_0_7"].astype(int)              # Ziel als Serie merken
    sex_series = df_ml["Sex_norm"].astype(str) if "Sex_norm" in df_ml else None


    # Ziel & Leckage-Spalten entfernen
    y = df_ml["AKI_any_0_7"].astype(int)
    leak_cols = [c for c in ["AKI_any_0_7", "highest_AKI_stage_0_7"] if c in df_ml.columns]
    X = df_ml.drop(columns=leak_cols, errors="ignore").copy()

    # Gewünschte numerische Prädiktoren (nur nehmen, wenn vorhanden)
    num_wanted = [
        "age_years_at_first_op", "n_ops", "duration_hours",
        "crea_baseline", "crea_peak_0_48", "crea_delta_0_48", "crea_rate_0_48",
        "cysc_baseline", "cysc_peak_0_48", "cysc_delta_0_48", "cysc_rate_0_48",
        "vis_max_0_24", "vis_mean_0_24", "vis_max_6_24", "vis_auc_0_24", "vis_auc_0_48",
    ]
    num_cols = [c for c in num_wanted if c in X.columns]

    # Kategoriale Prädiktoren (nur Sex_norm)
    cat_cols = [c for c in ["Sex_norm"] if c in X.columns]

    # Numerik wirklich numerisch machen
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # Unerwünschte object-Spalten (außer Sex_norm) droppen
    extra_obj = [c for c in X.columns if X[c].dtype == "object" and c not in cat_cols]
    if extra_obj:
        X = X.drop(columns=extra_obj)

    print("Numerische Features:", num_cols)
    print("Kategoriale Features:", cat_cols)

### 3\) Pipeline für Preprocessing und Modell

Es wird eine Scikit-Learn `Pipeline` erstellt, die das gesamte Machine-Learning-Setup kapselt:

1.  **`ColumnTransformer` (Preprocessing):**
      * **Numerische Features:** Fehlende Werte werden mit dem Median imputiert (`SimpleImputer`), gefolgt von einer Standardisierung (`StandardScaler`).
      * **Kategoriale Features:** Fehlende Werte werden mit dem Modus imputiert (`SimpleImputer`), gefolgt von einer One-Hot-Kodierung (`OneHotEncoder`).
2.  **Modell:** Ein `RandomForestClassifier` wird als Modell verwendet, mit einer angepassten Gewichtung der Klassen (`class_weight="balanced_subsample"`) zur Behandlung von Ungleichgewichten in der Zielvariable.

<!-- end list -->

In [ ]:
# =========================================
# 3) Pipeline (Impute/Scale/OneHot + Modell)
# =========================================
if adata is not None:
    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ],
        remainder="drop"
    )

    clf = Pipeline(steps=[
        ("prep", preprocess),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ))
    ])
else:
    clf = None

### 4\) Training, Test und Bewertung

Die Daten werden in Trainings- (75%) und Test-Sets (25%) aufgeteilt, wobei die Aufteilung **stratifiziert** erfolgt, um die Verteilung der Zielvariable in beiden Sets beizubehalten.

1.  Die Pipeline (`clf`) wird auf den Trainingsdaten trainiert.
2.  Vorhersagen und Wahrscheinlichkeiten werden auf den Testdaten generiert.
3.  Der **Classification Report** und die **ROC-AUC** (Area Under the Curve) werden auf den Testdaten ausgegeben.
4.  Optional wird eine 5-fache **Stratified Cross-Validation (CV)** über den gesamten Datensatz durchgeführt, um eine robustere Schätzung der Modellleistung zu erhalten.

<!-- end list -->

In [ ]:
# =========================
# 4) Train/Test + Bewertung
# =========================
if clf is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.25, random_state=42
    )

    # Training des Modells
    clf.fit(X_train, y_train)
    # Vorhersagen auf dem Test-Set
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    print("\n=== Classification Report (Test) ===")
    print(classification_report(y_test, y_pred, digits=3))
    print("ROC-AUC (Test):", roc_auc_score(y_test, y_proba))

    # (Optional) 5-fold CV auf Gesamtdaten
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_cv = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"CV ROC-AUC 5-fold: {auc_cv.mean():.3f} ± {auc_cv.std():.3f}")

### 5\) Feature-Importances

Die Wichtigkeit der Features aus dem trainierten Random Forest Modell wird extrahiert. Die Namen der Features werden nach der One-Hot-Kodierung der kategorialen Spalten rekonstruiert, und die Top 15 wichtigsten Features werden ausgegeben.

Anschließend wird die Zielvariable **AKI\_any\_0\_7** und gegebenenfalls **Sex\_norm** in das `adata.obs`-Feld verschoben. Dies ist wichtig, da `adata.var_names` nur Features enthalten soll, die in das Training einfließen, während `adata.obs` für Metadaten wie die Zielvariable oder Gruppierungsinformationen vorgesehen ist.

In [ ]:
# =========================
# 5) Feature-Importances
# =========================
if clf is not None:
    try:
        # Namen nach One-Hot rekonstruieren
        oh = clf.named_steps["prep"].named_transformers_["cat"].named_steps["oh"] if cat_cols else None
        num_names = num_cols
        cat_names = list(oh.get_feature_names_out(cat_cols)) if oh is not None else []
        feat_names = num_names + cat_names

        importances = clf.named_steps["rf"].feature_importances_
        order = np.argsort(importances)[::-1]
        top = min(15, len(importances))

        print("\nTop-Features:")
        for i in range(top):
            j = order[i]
            print(f"{i+1:>2}. {feat_names[j]}  |  {importances[j]:.4f}")
    except Exception as e:
        print("Feature-Importances nicht darstellbar:", e)

    # Label in obs schreiben (nachdem du es aus var_names entfernt hast)
    # a) sicherstellen, dass AKI_any_0_7 NICHT mehr in var_names ist
    if "AKI_any_0_7" in adata.var_names:
        adata = adata[:, [v for v in adata.var_names if v != "AKI_any_0_7"]].copy()

    # b) nun das zuvor gepufferte Label in obs spiegeln
    adata.obs["AKI_any_0_7"] = label_series.astype("category").values
    if sex_series is not None:
        adata.obs["Sex_norm"] = sex_series.astype("category").values

    # Kontrolle (sollte False / True drucken)
    print("'AKI_any_0_7' in var_names? ", "AKI_any_0_7" in list(adata.var_names))
    print("'AKI_any_0_7' in obs? ", "AKI_any_0_7" in adata.obs.columns)

### 6\) Plot-Galerie (ehrapy und Seaborn)

Dieser Abschnitt erstellt verschiedene Visualisierungen, um die Beziehungen zwischen ausgewählten Features und der Zielvariable **AKI\_any\_0\_7** zu untersuchen.

Zuerst wird sichergestellt, dass die Zielvariable und andere Gruppierungsvariablen nur in `adata.obs` und nicht in `adata.var_names` vorhanden sind, da dies für Visualisierungsfunktionen wie `ep.pl.violin` (Violinplot) erforderlich ist.

  * **Violinplot (ehrapy):** Zeigt die Verteilung des Alters (`age_years_at_first_op`) aufgeschlüsselt nach AKI-Status.
  * **Boxplots (Seaborn):** Zeigen die Verteilung der Operationsdauer (`duration_hours`) und der Veränderung des Kreatinin-Werts (`crea_delta_0_48`) im Verhältnis zum AKI-Status.

Alle erzeugten Diagramme werden im Ordner `Diagramme` gespeichert.

In [ ]:
# =========================================
# 6) Plot-Galerie (ehrapy) – mit Konflikt-Fix
# =========================================
if adata is not None:
    # a) Zielvariable darf NICHT in var_names stehen (sonst Konflikt in groupby)
    if "AKI_any_0_7" in adata.var_names:
        keep_vars = [v for v in adata.var_names if v != "AKI_any_0_7"]
        adata = adata[:, keep_vars].copy()

    # b) Ziel/Grouping in obs spiegeln
    df_all = adata.to_df()
    if "AKI_any_0_7" in df_all.columns:
        adata.obs["AKI_any_0_7"] = df_all["AKI_any_0_7"].astype("category")
    if "Sex_norm" in df_all.columns and "Sex_norm" not in adata.obs.columns:
        adata.obs["Sex_norm"] = df_all["Sex_norm"].astype("category")

    print("'AKI_any_0_7' in var_names? ", "AKI_any_0_7" in list(adata.var_names))
    print("'AKI_any_0_7' in obs? ", "AKI_any_0_7" in adata.obs.columns)


    # --- sauberes DF für Plots zusammenstellen ---
    df_all = adata.to_df().copy()

    # Label/Grouping aus obs ergänzen (ohne Spaltenkollision)
    if "AKI_any_0_7" in adata.obs.columns:
        df_all["AKI_any_0_7"] = adata.obs["AKI_any_0_7"].astype(int).values
    if ("Sex_norm" not in df_all.columns) and ("Sex_norm" in adata.obs.columns):
        df_all["Sex_norm"] = adata.obs["Sex_norm"].astype(str).values

    # Nur Zeilen mit benötigten Spalten behalten
    df_box1 = df_all[["AKI_any_0_7", "duration_hours"]].dropna()
    df_box2 = df_all[["AKI_any_0_7", "crea_delta_0_48"]].dropna()

    import seaborn as sns
    import matplotlib.pyplot as plt
    import os
    os.makedirs("Diagramme", exist_ok=True)

    # --- Boxplot: OP-Dauer nach AKI ---
    plt.figure(figsize=(5,4))
    sns.boxplot(x="AKI_any_0_7", y="duration_hours", data=df_box1, showfliers=False)
    sns.stripplot(x="AKI_any_0_7", y="duration_hours", data=df_box1, color="0.3", size=2, alpha=0.35)
    plt.title("OP-Dauer nach AKI (0–7 Tage)")
    plt.xlabel("AKI 0–7 Tage (0/1)")
    plt.ylabel("Dauer (h)")
    plt.tight_layout()
    plt.savefig("Diagramme/box_duration_AKI.png", dpi=300)
    plt.close()

    # --- Boxplot: Kreatinin-Delta 0–48h nach AKI ---
    plt.figure(figsize=(5,4))
    sns.boxplot(x="AKI_any_0_7", y="crea_delta_0_48", data=df_box2, showfliers=False)
    sns.stripplot(x="AKI_any_0_7", y="crea_delta_0_48", data=df_box2, color="0.3", size=2, alpha=0.35)
    plt.title("Kreatinin-Δ (0–48 h) nach AKI")
    plt.xlabel("AKI 0–7 Tage (0/1)")
    plt.ylabel("Δ Kreatinin (Einheit)")
    plt.tight_layout()
    plt.savefig("Diagramme/box_crea_delta_AKI.png", dpi=300)
    plt.close()

    # ----------- Violinplot (ehrapy) -----------
    try:
        ep.pl.violin(adata, keys=["age_years_at_first_op"], groupby="AKI_any_0_7")
        plt.title("Altersverteilung nach AKI (0–7 Tage)")
        plt.savefig("Diagramme/violin_age_AKI.png", dpi=300)
        plt.close()
        print("✅ Violinplot gespeichert: Diagramme/violin_age_AKI.png")
    except Exception as e:
        print("Violinplot-Fehler:", e)

    # ----------- Boxplots (Seaborn) -----------
    # Erneute Speicherung der Boxplots, um sicherzustellen, dass die Dateien da sind (Code-Redundanz aus dem Original)
    df_all = adata.to_df().copy()
    if "AKI_any_0_7" in adata.obs.columns:
        df_all["AKI_any_0_7"] = adata.obs["AKI_any_0_7"].astype(int).values

    # OP-Dauer nach AKI
    plt.figure(figsize=(5,4))
    sns.boxplot(x="AKI_any_0_7", y="duration_hours", data=df_all, showfliers=False)
    sns.stripplot(x="AKI_any_0_7", y="duration_hours", data=df_all,
                  color="0.3", size=2, alpha=0.35)
    plt.title("OP-Dauer nach AKI (0–7 Tage)")
    plt.savefig("Diagramme/box_duration_AKI.png", dpi=300)
    plt.close()

    # Kreatinin-Delta nach AKI
    plt.figure(figsize=(5,4))
    sns.boxplot(x="AKI_any_0_7", y="crea_delta_0_48", data=df_all, showfliers=False)
    sns.stripplot(x="AKI_any_0_7", y="crea_delta_0_48", data=df_all,
                  color="0.3", size=2, alpha=0.35)
    plt.title("Kreatinin-Δ (0–48 h) nach AKI")
    plt.savefig("Diagramme/box_crea_delta_AKI.png", dpi=300)
    plt.close()

    print("\n✅ Alle Diagramme gespeichert im Ordner 'Diagramme'.")